In [0]:
%pip install -U mlflow python-dotenv openai tiktoken langchain-openai langchain-chroma langchain-huggingface langchain-community langchain-text-splitters scikit-learn plotly chromadb sentence-transformers
dbutils.library.restartPython()

## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [0]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [0]:
# Databricks AI Gateway setup (free tier alternative)
# Get Databricks token - try environment variable or use notebook context
import os

try:
    # Try to get from dbutils (available in Databricks notebooks)
    databricks_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
except:
    # Fallback to environment variable
    databricks_token = os.environ.get("DATABRICKS_TOKEN", "dummy-token")

client = OpenAI(
    api_key=databricks_token,
    base_url="https://ai-gateway.cloud.databricks.com/v1"
)

In [0]:
# price is a factor for our company, so we're going to use a low cost model

db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    MODEL = "gpt-4o-mini"
    openai = OpenAI()
else:
    print("OpenAI API Key not set - using Databricks AI Gateway")
    MODEL = "databricks-gpt-oss-120b"  # Databricks free tier model
    openai = client  # Uses the Databricks client from previous cell

In [0]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

In [0]:
# How many tokens in all the documents?

try:
    # Try to get the encoding for the specific model (works for OpenAI models)
    encoding = tiktoken.encoding_for_model(MODEL)
except KeyError:
    # For non-OpenAI models (like Databricks), use cl100k_base encoding
    # This is the encoding used by GPT-4 and is a good approximation
    encoding = tiktoken.get_encoding("cl100k_base")
    
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

In [0]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    print(doc_type)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

In [0]:
documents[0]

In [0]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

In [0]:
chunks[100]

In [0]:
len(chunks)

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [0]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [0]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

#### Understanding Vector Store Operations

**The Vector Store Architecture:**

When you create a vector store with `Chroma.from_documents()`, you get two levels of abstraction:

1. **`vectorstore`** - The LangChain wrapper that provides a high-level interface for:
   - `.similarity_search()` - Query for similar documents
   - `.as_retriever()` - Convert to a retriever for RAG chains
   - `.add_documents()` - Add new documents
   - `.delete()` - Remove documents

2. **`vectorstore._collection`** - The underlying Chroma collection object that gives direct access to:
   - `.count()` - Number of vectors stored
   - `.get()` - Retrieve raw vectors, documents, and metadata
   - `.query()` - Low-level vector similarity queries
   - `.peek()` - Quick preview of stored data

**Typical Vector Store Workflow:**

```
1. CREATE → Load documents and create embeddings
2. INSPECT → Check vector count, dimensions, sample data
3. QUERY → Search for relevant chunks (similarity_search, MMR, etc.)
4. RETRIEVE → Use as retriever in RAG chains with LLMs
5. MAINTAIN → Add/update/delete documents as data changes
```

---

**The `.get()` Method:**

`collection.get(include=['embeddings', 'documents', 'metadatas'])` retrieves stored data with fine-grained control:

* **`include` parameter** - What to retrieve:
  - `'embeddings'` - The actual vector arrays (384 dimensions in our case)
  - `'documents'` - The text content of each chunk
  - `'metadatas'` - Associated metadata (doc_type, source, etc.)
  - `'ids'` - Unique identifiers for each chunk

* **Common patterns:**
  ```python
  # Get everything
  result = collection.get(include=['embeddings', 'documents', 'metadatas'])
  
  # Get only metadata (fast, no large vectors)
  result = collection.get(include=['metadatas'])
  
  # Get specific documents by ID
  result = collection.get(ids=['id1', 'id2'], include=['documents'])
  
  # Limit results
  result = collection.get(limit=10, include=['documents'])
  ```

**Why Access `_collection` Directly?**

* **Visualization**: Need raw embeddings to plot vectors (like we do in Part C)
* **Debugging**: Inspect what's actually stored vs. what queries return
* **Batch operations**: More efficient for bulk data retrieval
* **Statistics**: Count vectors, check dimensions, analyze metadata distribution
* **Low-level control**: Advanced operations not exposed by the LangChain wrapper

**Note:** The underscore prefix (`_collection`) indicates it's a private attribute. Use it carefully and prefer the high-level `vectorstore` methods when possible.

---

**Query Methods (Used in Part D):**

The `vectorstore` object provides several high-level methods for querying:

1. **`.similarity_search(query, k)`**
   - Returns the top `k` most similar document chunks
   - Uses vector distance (cosine similarity or L2) to rank results
   - Returns: List of Document objects
   - Example: `results = vectorstore.similarity_search("What products exist?", k=3)`

2. **`.similarity_search_with_score(query, k)`**
   - Same as similarity_search but includes distance scores
   - Returns: List of (Document, score) tuples
   - Lower scores = more similar (closer in vector space)
   - Example: `results = vectorstore.similarity_search_with_score("query", k=3)`
   - Useful for: Setting confidence thresholds, filtering low-quality results

3. **`.as_retriever(search_type, search_kwargs)`**
   - Converts vectorstore to a Retriever object for LangChain
   - Standardized interface that works with LLM chains
   - Returns: Retriever object with `.invoke()` method
   - Example:
     ```python
     retriever = vectorstore.as_retriever(
         search_type="similarity",
         search_kwargs={"k": 3}
     )
     results = retriever.invoke("query")
     ```
   - Used in: Production RAG systems, question-answering chains

4. **`.max_marginal_relevance_search(query, k, fetch_k)`**
   - Balances relevance with diversity to avoid redundant results
   - First fetches `fetch_k` candidates, then selects diverse `k` final results
   - Returns: List of Document objects (more diverse than similarity_search)
   - Example: `results = vectorstore.max_marginal_relevance_search(query, k=4, fetch_k=10)`
   - Useful for: Getting broader context from multiple sources

5. **Metadata Filtering** (parameter on search methods)
   - Add `filter={"key": "value"}` to any search method
   - Pre-filters at database level before searching
   - Example: `vectorstore.similarity_search(query, k=3, filter={"doc_type": "employees"})`
   - Useful for: Scoping searches, multi-tenant systems, category-specific queries

---

**Understanding `enumerate()` (Used Throughout Part D):**

The `enumerate()` function is a Python built-in that adds automatic counters to iterables:

**Basic syntax:**
```python
for i, item in enumerate(items, start=1):
    # i is the counter (starting at 1)
    # item is the actual element from the list
```

**Why use it?**
* Eliminates manual counter management (no need for `i = 0; i += 1`)
* Makes code cleaner and more Pythonic
* Allows custom starting values (default is 0, we use 1 for human-readable numbering)

**Examples from Part D:**

```python
# Without enumerate (manual counter)
i = 1
for doc in results:
    print(f"Result {i}:")
    i += 1

# With enumerate (automatic counter starting at 1)
for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
```

**With tuples (similarity_search_with_score):**
```python
for i, (doc, score) in enumerate(results, 1):
    # i = counter (1, 2, 3, ...)
    # doc = Document object
    # score = similarity score
    print(f"Result {i} (Score: {score:.4f}):")
```

**Key points:**
* `enumerate(items)` starts counting at 0
* `enumerate(items, 1)` starts counting at 1 (better for display)
* Works with any iterable (lists, tuples, query results, etc.)
* Can unpack tuples: `enumerate([(doc1, score1), (doc2, score2)], 1)` → `(i, (doc, score))`

### Part C: Visualize!

In [0]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [0]:
result

In [0]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [0]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

### Part D: Query the Vector Store

Now that we have our vector store built, let's explore different ways to query it.

#### 1. Basic Similarity Search

**What it does:**
The `similarity_search()` method finds the `k` most similar document chunks to your query by:
1. Converting your query text into a 384-dimensional embedding vector using the same model (all-MiniLM-L6-v2)
2. Computing the distance between your query vector and all 413 chunk vectors in the database
3. Returning the top `k` closest matches based on vector distance (typically cosine similarity or L2 distance)

**Why use it:**
- **Simplest and fastest** query method
- Perfect for quick lookups when you just need relevant context
- Returns only the documents without extra metadata
- Ideal for prototyping and testing your RAG system

**Use case:** When you need quick, straightforward answers and don't need to see confidence scores or fine-tune retrieval parameters.

#### 2. Similarity Search with Scores

**What it does:**
Identical to basic similarity search, but also returns a **distance score** for each result:
- **Lower scores** = more similar (closer in vector space)
- **Higher scores** = less similar (farther apart)
- The score is the actual distance metric (e.g., Euclidean distance) between vectors

**Why use it:**
- **Confidence assessment**: See how confident the retrieval system is
- **Threshold filtering**: Filter out results below a certain relevance threshold
- **Debugging**: Understand if your embeddings are working well
- **Quality control**: Identify when no good matches exist (all scores are high)

**Use case:** When you need to set quality thresholds in production (e.g., "only use results with score < 1.5") or want to understand retrieval quality during development.

#### 3. Retriever Interface for RAG Chains

**What it does:**
Converts your vector store into a **Retriever** object with a standardized interface:
- Uses the `.invoke()` method (standard across LangChain)
- Returns documents in the format expected by LangChain chains
- Allows configuration of search type and parameters via `search_kwargs`

**Why use it:**
- **LangChain integration**: Required when building RAG chains with LLMs
- **Standardization**: Works seamlessly with other LangChain components (prompts, chains, agents)
- **Composability**: Can be easily swapped with other retrievers without changing downstream code
- **Production-ready**: The standard pattern for building RAG applications

**Use case:** When building a complete question-answering system where retrieved documents are passed to an LLM for answer generation. This is the interface you'll use in production RAG systems.

#### 4. Max Marginal Relevance (MMR)

**What it does:**
Balances **relevance** with **diversity** using a two-step process:
1. First fetches `fetch_k` candidate documents (e.g., 10) based on similarity
2. Then iteratively selects `k` final results that are:
   - Relevant to the query
   - Diverse from each other (not redundant)

**The algorithm:**
- Selects the most relevant document first
- For each subsequent selection, picks documents that maximize: `λ × similarity_to_query - (1-λ) × similarity_to_selected`
- Default λ ≈ 0.5 balances relevance and diversity equally

**Why use it:**
- **Avoid redundancy**: Prevents retrieving 3 nearly-identical chunks from the same document
- **Broader context**: Gets information from multiple sources/perspectives
- **Better answers**: LLMs perform better with diverse context vs. repetitive information

**Use case:** When your documents contain redundant information or you want to give the LLM a broader view of the topic (e.g., product features across different product docs, employee info from different offices).

#### 5. Metadata Filtering

**What it does:**
Applies a **pre-filter** before similarity search:
1. Filters the vector database to only chunks matching the metadata criteria (e.g., `doc_type='employees'`)
2. Then performs similarity search only within that filtered subset
3. More efficient than post-filtering because it searches a smaller space

**Why use it:**
- **Scoped search**: Restrict search to specific document types, dates, authors, etc.
- **Improved precision**: Eliminate irrelevant categories from consideration
- **Cost reduction**: Search fewer vectors = faster queries and lower compute
- **User control**: Let users specify what types of documents to search

**Technical note:**
The filter is applied at the database level (in Chroma), not after retrieval, making it efficient even with large collections.

**Use case:** When you know the answer should come from a specific category (e.g., "employee benefits" should only search employee docs, not product specs), or when building multi-tenant systems where each user should only see their own data.

In [0]:
# 1. Basic Similarity Search - Returns the k most similar documents

query = "What insurance products does Insurellm offer?"
results = vectorstore.similarity_search(query, k=3)
print(result)


print(f"Query: {query}\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Source: {doc.metadata['source']}")
    print(f"Type: {doc.metadata['doc_type']}")
    print("-" * 80)

In [0]:
# 2. Similarity Search with Scores - Shows how relevant each result is
# Lower scores = more similar (closer distance in vector space)

query = "Who are the employees in the San Francisco office?"
results = vectorstore.similarity_search_with_score(query, k=3)

print(f"Query: {query}\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"Result {i} (Score: {score:.4f}):")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Source: {doc.metadata['source']}")
    print(f"Type: {doc.metadata['doc_type']}")
    print("-" * 80)

In [0]:
# 3. As a Retriever - Useful for LangChain RAG chains
# The retriever interface is standardized for use with LLMs

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

query = "What is the company culture like at Insurellm?"
results = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Source: {doc.metadata['source']}")
    print("-" * 80)

In [0]:
# 4. Max Marginal Relevance (MMR) - Balances relevance with diversity
# Useful when you want varied results, not just the most similar ones

query = "Tell me about Insurellm's products"
results = vectorstore.max_marginal_relevance_search(query, k=4, fetch_k=10)

print(f"Query: {query}\n")
print("MMR returns diverse results to avoid redundancy:\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"Content: {doc.page_content[:150]}...")
    print(f"Source: {doc.metadata['source']}")
    print(f"Type: {doc.metadata['doc_type']}")
    print("-" * 80)

In [0]:
# 5. Filter by Metadata - Query specific document types
# Useful when you want to search only within certain categories

query = "What are the employee benefits?"

# Search only in employee documents
results = vectorstore.similarity_search(
    query, 
    k=3,
    filter={"doc_type": "employees"}
)

print(f"Query: {query}")
print(f"Filtering by: doc_type='employees'\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Source: {doc.metadata['source']}")
    print("-" * 80)